# 💬 템플릿 5 — 카톡 대화 분석기

카톡 대화 export 파일(.txt)을 분석해서 통계 + AI 인사이트.

## 카톡 export 방법
1. 카톡 → 대상 채팅방 진입
2. 우측 상단 ☰ 메뉴 → "대화 내용 내보내기"
3. .txt 파일 받기 → 이 노트북에 업로드

## 분석 내용
- 누가 가장 활발한가
- 대화의 전체적 분위기 (AI)
- 자주 등장하는 키워드
- 시간대별 분포

## ⚠️ 프라이버시
본인 동의 받은 대화만 분석하세요. 결과를 외부에 공유 시 이름은 가리기.

In [ ]:
!pip install -q gradio openai matplotlib

In [ ]:
SERVER_URL = "https://YOUR_URL.trycloudflare.com".strip().rstrip("/")
MODEL = "qwen2.5:7b-instruct"
assert "YOUR" not in SERVER_URL, "❌ SERVER_URL을 강사가 알려준 URL로 바꾸세요"

import httpx
from openai import OpenAI
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key="ollama",
                http_client=httpx.Client(headers={"User-Agent": "Mozilla/5.0"}))

# 한글 폰트 설정
!apt-get install -qq -y fonts-nanum > /dev/null 2>&1
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
print("✅ 준비 완료")

In [ ]:
import gradio as gr
import re
import pandas as pd
from collections import Counter

# 샘플 데이터 (파일 없을 때 데모용)
SAMPLE = """[김민수] [오후 1:00] 야 뭐해
[이지은] [오후 1:01] 그냥 누워있어 ㅋㅋ
[김민수] [오후 1:01] 나도ㅋㅋㅋ
[이지은] [오후 1:02] 오늘 저녁 뭐먹을지 고민중
[박철수] [오후 1:05] 그러면 우리 치킨 시킬까?
[김민수] [오후 1:05] 콜
[이지은] [오후 1:06] 좋아!!
[박철수] [오후 1:10] 뭐 시킬건데
[김민수] [오후 1:11] BBQ 황금올리브
[이지은] [오후 1:11] 마라치킨도 어때?
[박철수] [오후 1:12] 마라 ㄱㄱ
[김민수] [오후 1:30] 시켰어 30분 걸린대
[이지은] [오후 2:00] 도착했어?
[김민수] [오후 2:01] ㅇㅇ 지금 먹는중
[박철수] [오후 2:10] 맛있다 ㄹㅇ"""

def parse_katalk(text):
    """카톡 모바일/PC export 두 가지 형식 지원"""
    rows = []
    
    # 형식 1 모바일: [이름] [오후 1:00] 메시지
    p1 = re.compile(r'\[([^\]]+)\]\s*\[(오전|오후)\s*(\d+):(\d+)\]\s*(.+)')
    # 형식 2 PC: 2024. 11. 20. 오후 1:00, 이름 : 메시지
    p2 = re.compile(r'(\d{4})\.\s*(\d+)\.\s*(\d+)\.\s*(오전|오후)\s*(\d+):(\d+),\s*([^:]+):\s*(.+)')
    
    for line in text.split("\n"):
        m1 = p1.match(line.strip())
        if m1:
            name, ampm, h, mm, msg = m1.groups()
            hour = int(h) + (12 if ampm == "오후" and int(h) != 12 else 0)
            rows.append({"name": name.strip(), "hour": hour, "msg": msg.strip()})
            continue
        m2 = p2.match(line.strip())
        if m2:
            yr, mo, d, ampm, h, mm, name, msg = m2.groups()
            hour = int(h) + (12 if ampm == "오후" and int(h) != 12 else 0)
            rows.append({"name": name.strip(), "hour": hour, "msg": msg.strip()})
    return pd.DataFrame(rows)

def analyze(file_obj):
    if file_obj is None:
        text = SAMPLE
        note = "ℹ️ 파일이 없어서 샘플 데이터로 분석합니다"
    else:
        with open(file_obj.name, "r", encoding="utf-8") as f:
            text = f.read()
        note = ""
    
    df = parse_katalk(text)
    if len(df) == 0:
        return "❌ 파싱 실패: 카톡 모바일/PC 표준 형식이 아닐 수 있어요", None, None, ""
    
    # 1) 통계
    by_name = df["name"].value_counts()
    stats = f"{note}\n\n### 📊 기본 통계\n"
    stats += f"- 총 메시지: **{len(df):,}개**\n"
    stats += f"- 참여자: **{len(by_name)}명**\n\n"
    stats += "### 🏆 활발한 사람 Top 3\n"
    for i, (name, n) in enumerate(by_name.head(3).items(), 1):
        medal = ["🥇", "🥈", "🥉"][i-1]
        stats += f"{medal} **{name}** — {n}개 ({n/len(df)*100:.1f}%)\n"
    
    # 2) 사람별 그래프
    fig1, ax1 = plt.subplots(figsize=(7, 4))
    by_name.head(10).plot(kind="barh", ax=ax1, color="steelblue")
    ax1.set_title("참여자별 메시지 수")
    ax1.invert_yaxis()
    plt.tight_layout()
    
    # 3) 시간대 그래프
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    df["hour"].value_counts().sort_index().reindex(range(24), fill_value=0).plot(
        kind="bar", ax=ax2, color="coral"
    )
    ax2.set_title("시간대별 메시지 수")
    ax2.set_xlabel("시간 (0~23)")
    plt.tight_layout()
    
    # 4) AI 분석
    sample_msgs = df.sample(min(40, len(df)))
    sample_text = "\n".join(f"{r['name']}: {r['msg']}" for _, r in sample_msgs.iterrows())
    
    prompt = f"""다음은 카톡 대화 일부입니다 (총 {len(df)}개 중 일부 발췌).

[대화]
{sample_text[:3000]}

다음을 한국어로 분석해주세요:
1. **전체 분위기** (한 줄로)
2. **자주 등장하는 화제** (3개)
3. **각 참여자 대화 스타일 특징** (이름별 한 줄)
4. **재밌는 관찰 포인트** (1개)"""
    
    ai_analysis = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600, temperature=0.5,
    ).choices[0].message.content
    
    return stats, fig1, fig2, f"### 🤖 AI 인사이트\n\n{ai_analysis}"

with gr.Blocks(title="💬 카톡 분석", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 💬 카톡 대화 분석기")
    gr.Markdown("📂 파일 없으면 샘플로 데모됩니다. 본인 카톡 export(.txt) 업로드하면 진짜 분석!")
    
    with gr.Row():
        file = gr.File(label="카톡 .txt 파일", file_types=[".txt"])
        btn = gr.Button("➤ 분석 시작", variant="primary", scale=2)
    
    with gr.Row():
        stats = gr.Markdown()
        ai = gr.Markdown()
    
    with gr.Row():
        plot1 = gr.Plot()
        plot2 = gr.Plot()
    
    btn.click(analyze, file, [stats, plot1, plot2, ai])

demo.launch(share=True)

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움
- 요일별 분석 추가
- 가장 긴 메시지 / 가장 짧은 메시지
- 이모티콘 사용 통계
- 가장 자주 쓴 단어 Top 20

### 중간
- 워드클라우드 시각화
- 친밀도 점수 (메시지 양 + 답변 속도)
- 사람별 감정 점수 (긍정/부정 단어 비율)
- "이 사람의 톡 스타일을 한 줄로" AI 묘사

### 도전적
- 대화 흐름을 타임라인으로 (시간 ↔ 분위기)
- 카톡 학습 LLM (본인 톡 스타일로 답하는 봇)
- 친구별 케미스트리 매칭 (커플 / 절친 / 적정거리)
- 연도별 비교 (작년 vs 올해 대화 패턴)

### 🎁 자랑하기 팁
- "친구야 이거 봐봐 우리 단톡방 분석" 인증샷 SNS 공유
- 1년치 카톡 정리해서 "올해의 카톡 리포트" 만들기